## Pipeline CI/CD — Dataset confiable de teléfonos de clientes

## 0. Parámetros (celda `parameters` de papermill)

In [ ]:
ENTORNO = "dev"
RUTA_CONFIG = "../conf/config.yml"
RUTA_GATES = "../conf/quality_gates.yml"
DIR_ENTRADA = "../data/samples"
DIR_SALIDA = "../data/gold"
VERSION_DATASET = "1.0.0"
PUBLICAR = False
GIT_SHA = "local"
EMITIR_OBSERVATORIO = True
DIR_OBSERVATORIO = "../data/observatorio"
RUTA_ESTADOS = "../data/observatorio/estados_verificados.parquet"

## 1. Configuración, dependencias y contexto de ejecución

In [ ]:
from __future__ import annotations

import hashlib
import hmac
import json
import logging
import os
import re
import unicodedata
from dataclasses import dataclass, asdict
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
import phonenumbers
from phonenumbers import PhoneNumberFormat, PhoneNumberType, NumberParseException
from phonenumbers import carrier, geocoder, timezone as ph_timezone

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
)
log = logging.getLogger("phones")

@dataclass(frozen=True)
class ConfigPipeline:

    region_default: str = "CO"
    regiones_permitidas: tuple[str, ...] = ("CO", "MX", "US", "ES", "AR", "CL", "PE")
    tipos_contactables: tuple[str, ...] = ("MOBILE", "FIXED_LINE_OR_MOBILE")
    idioma_geo: str = "es"
    dias_vigencia_consentimiento: int = 730
    max_telefonos_por_cliente: int = 3
    env_clave_hmac: str = "PHONE_HMAC_KEY"

    @classmethod
    def desde_yaml(cls, ruta: str | Path) -> "ConfigPipeline":
        import yaml

        datos = yaml.safe_load(Path(ruta).read_text(encoding="utf-8")) or {}
        campos = {f: datos[f] for f in cls.__dataclass_fields__ if f in datos}
        for k, v in campos.items():
            if isinstance(v, list):
                campos[k] = tuple(v)
        return cls(**campos)

In [ ]:
import yaml

CONF_RAW = yaml.safe_load(Path(RUTA_CONFIG).read_text(encoding="utf-8")) if Path(RUTA_CONFIG).exists() else {}
CFG = ConfigPipeline.desde_yaml(RUTA_CONFIG) if Path(RUTA_CONFIG).exists() else ConfigPipeline()
GATES = (
    yaml.safe_load(Path(RUTA_GATES).read_text(encoding="utf-8"))
    if Path(RUTA_GATES).exists()
    else {}
)
EJECUCION = {
    "entorno": ENTORNO,
    "version_dataset": VERSION_DATASET,
    "git_sha": GIT_SHA,
    "inicio_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
print(json.dumps(EJECUCION, indent=2, ensure_ascii=False))
CFG

## 2. Ingesta — capa bronze

In [ ]:
def hash_fila(fila: pd.Series) -> str:
    payload = "|".join(f"{k}={fila[k]}" for k in sorted(fila.index))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]

def leer_fuente(spec: dict[str, Any], dir_base: str | Path) -> pd.DataFrame:
    ruta = Path(dir_base) / spec["archivo"]
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, na_values=[""])
    faltantes = set(spec["columnas_tel"] + [spec["columna_id"]]) - set(df.columns)
    if faltantes:
        raise ValueError(f"{ruta.name}: faltan columnas obligatorias {sorted(faltantes)}")
    df["_fuente"] = spec["nombre"]
    df["_prioridad_fuente"] = int(spec["prioridad"])
    df["_ingerido_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    df["_hash_fila"] = df.apply(hash_fila, axis=1)
    log.info("Fuente '%s': %d filas leídas de %s", spec["nombre"], len(df), ruta.name)
    return df

def a_formato_largo(df: pd.DataFrame, spec: dict[str, Any]) -> pd.DataFrame:
    fijas = [c for c in df.columns if c not in spec["columnas_tel"]]
    largo = df.melt(
        id_vars=fijas,
        value_vars=spec["columnas_tel"],
        var_name="campo_origen",
        value_name="telefono_crudo",
    )
    largo = largo[largo["telefono_crudo"].notna()].copy()
    largo = largo.rename(columns={spec["columna_id"]: "cliente_id"})
    return largo.reset_index(drop=True)

def ingerir(fuentes: Iterable[dict[str, Any]], dir_base: str | Path) -> pd.DataFrame:
    partes = [a_formato_largo(leer_fuente(s, dir_base), s) for s in fuentes]
    bronze = pd.concat(partes, ignore_index=True, sort=False)
    log.info("Bronze consolidado: %d candidatos de %d fuentes", len(bronze), len(partes))
    return bronze

In [ ]:
FUENTES = CONF_RAW.get("fuentes") or [
    {
        "nombre": "crm",
        "archivo": "crm_clientes.csv",
        "columna_id": "cliente_id",
        "columnas_tel": ["telefono_movil", "telefono_fijo"],
        "prioridad": 1,
    },
    {
        "nombre": "ecommerce",
        "archivo": "ecommerce_perfiles.csv",
        "columna_id": "cliente_id",
        "columnas_tel": ["celular_checkout"],
        "prioridad": 2,
    },
]

bronze = ingerir(FUENTES, DIR_ENTRADA)
bronze.head(10)

## 3. Normalización y validación sintáctica (E.164)

In [ ]:
_RE_EXTENSION = re.compile(
    r"(?:\s*(?:ext|extensi[oó]n|anexo|x)\s*[:.\-]?\s*)(\d{1,6})\s*$", re.IGNORECASE
)
_RE_BASURA = re.compile(r"[^\d+]")
_RE_REPETIDOS = re.compile(r"^(\d)\1{6,}$")
_SECUENCIAS_PRUEBA = {"1234567890", "0123456789", "9876543210"}

_TIPOS = {
    PhoneNumberType.MOBILE: "MOBILE",
    PhoneNumberType.FIXED_LINE: "FIXED_LINE",
    PhoneNumberType.FIXED_LINE_OR_MOBILE: "FIXED_LINE_OR_MOBILE",
    PhoneNumberType.TOLL_FREE: "TOLL_FREE",
    PhoneNumberType.PREMIUM_RATE: "PREMIUM_RATE",
    PhoneNumberType.VOIP: "VOIP",
    PhoneNumberType.SHARED_COST: "SHARED_COST",
    PhoneNumberType.PAGER: "PAGER",
    PhoneNumberType.PERSONAL_NUMBER: "PERSONAL_NUMBER",
    PhoneNumberType.UAN: "UAN",
    PhoneNumberType.VOICEMAIL: "VOICEMAIL",
}

def limpiar_crudo(valor: Any) -> tuple[str, str | None]:
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return "", None
    texto = unicodedata.normalize("NFKD", str(valor)).strip()
    extension = None
    m = _RE_EXTENSION.search(texto)
    if m:
        extension = m.group(1)
        texto = texto[: m.start()]
    if re.match(r"^00[1-9]", texto):
        texto = "+" + texto[2:]
    limpio = _RE_BASURA.sub("", texto)
    if "+" in limpio[1:]:
        limpio = limpio[0] + limpio[1:].replace("+", "")
    return limpio, extension

def es_sospechoso(nacional: str) -> bool:
    return bool(_RE_REPETIDOS.match(nacional)) or nacional in _SECUENCIAS_PRUEBA

def normalizar_telefono(valor: Any, cfg: ConfigPipeline) -> dict[str, Any]:
    base: dict[str, Any] = {
        "e164": None, "extension": None, "valido": False, "motivo_rechazo": "VACIO",
        "tipo_linea": None, "region": None, "operador": None, "zona_horaria": None,
        "contactable": False,
    }
    limpio, extension = limpiar_crudo(valor)
    base["extension"] = extension
    if not re.sub(r"\D", "", limpio):
        return base

    region = None if limpio.startswith("+") else cfg.region_default
    try:
        numero = phonenumbers.parse(limpio, region)
    except NumberParseException:
        return {**base, "motivo_rechazo": "NO_PARSEABLE"}

    if not phonenumbers.is_possible_number(numero):
        return {**base, "motivo_rechazo": "IMPOSIBLE"}
    if not phonenumbers.is_valid_number(numero):
        return {**base, "motivo_rechazo": "INVALIDO"}

    region_detectada = phonenumbers.region_code_for_number(numero)
    if region_detectada not in cfg.regiones_permitidas:
        return {**base, "motivo_rechazo": "REGION_NO_PERMITIDA", "region": region_detectada}
    if es_sospechoso(str(numero.national_number)):
        return {**base, "motivo_rechazo": "SOSPECHOSO_PRUEBA", "region": region_detectada}

    tipo = _TIPOS.get(phonenumbers.number_type(numero), "DESCONOCIDO")
    zonas = ph_timezone.time_zones_for_number(numero)
    return {
        "e164": phonenumbers.format_number(numero, PhoneNumberFormat.E164),
        "extension": extension,
        "valido": True,
        "motivo_rechazo": "OK",
        "tipo_linea": tipo,
        "region": region_detectada,
        "operador": carrier.name_for_number(numero, cfg.idioma_geo) or None,
        "zona_horaria": zonas[0] if zonas else None,
        "contactable": tipo in cfg.tipos_contactables,
    }

def normalizar_df(df: pd.DataFrame, cfg: ConfigPipeline, col: str = "telefono_crudo") -> pd.DataFrame:
    cache: dict[str, dict[str, Any]] = {}

    def _norm(v: Any) -> dict[str, Any]:
        clave = str(v)
        if clave not in cache:
            cache[clave] = normalizar_telefono(v, cfg)
        return cache[clave]

    enriquecido = pd.DataFrame([_norm(v) for v in df[col]], index=df.index)
    return pd.concat([df, enriquecido], axis=1)

In [ ]:
silver = normalizar_df(bronze, CFG)
resumen_motivos = (
    silver["motivo_rechazo"].value_counts(dropna=False).rename_axis("motivo").to_frame("filas")
)
resumen_motivos["%"] = (100 * resumen_motivos["filas"] / len(silver)).round(2)
resumen_motivos

In [ ]:
silver.loc[
    :, ["cliente_id", "_fuente", "telefono_crudo", "e164", "tipo_linea", "region",
        "operador", "contactable", "motivo_rechazo"]
].head(12)

## 4. Gobernanza: consentimiento, listas de exclusión y PII

In [ ]:
def obtener_clave_hmac(cfg: ConfigPipeline) -> bytes:
    clave = os.environ.get(cfg.env_clave_hmac)
    if not clave:
        raise RuntimeError(
            f"Falta la variable {cfg.env_clave_hmac}. Debe inyectarse desde el "
            "gestor de secretos (GitHub OIDC / Vault), nunca desde el repositorio."
        )
    return clave.encode("utf-8")

def seudonimizar(e164: Any, clave: bytes) -> str | None:
    if not isinstance(e164, str) or not e164:
        return None
    return hmac.new(clave, e164.encode("utf-8"), hashlib.sha256).hexdigest()

def enmascarar(e164: Any) -> str | None:
    if not isinstance(e164, str) or len(e164) < 8:
        return None
    return f"{e164[:6]}{'*' * (len(e164) - 8)}{e164[-2:]}"

def aplicar_consentimiento(
    df: pd.DataFrame, consentimientos: pd.DataFrame, dnc: pd.DataFrame, cfg: ConfigPipeline,
    hoy: date | None = None,
) -> pd.DataFrame:
    hoy = hoy or date.today()
    out = df.merge(consentimientos, on="cliente_id", how="left")

    fecha = pd.to_datetime(out.get("fecha_consentimiento"), errors="coerce")
    antiguedad = (pd.Timestamp(hoy) - fecha).dt.days
    opt_in = out.get("opt_in", pd.Series(index=out.index, dtype="object")).astype(str).str.lower()

    out["consentimiento_vigente"] = (
        opt_in.isin({"true", "1", "si", "sí", "yes"})
        & fecha.notna()
        & (antiguedad <= cfg.dias_vigencia_consentimiento)
    )
    out["dias_desde_consentimiento"] = antiguedad

    bloqueados = set(dnc["e164"].dropna()) if not dnc.empty else set()
    out["en_dnc"] = out["e164"].isin(bloqueados)

    out["apto_contacto"] = (
        out["valido"] & out["contactable"] & out["consentimiento_vigente"] & ~out["en_dnc"]
    )
    out["motivo_no_contacto"] = "OK"
    out.loc[~out["valido"], "motivo_no_contacto"] = out["motivo_rechazo"]
    out.loc[out["valido"] & ~out["contactable"], "motivo_no_contacto"] = "TIPO_NO_CONTACTABLE"
    out.loc[out["valido"] & ~out["consentimiento_vigente"], "motivo_no_contacto"] = "SIN_CONSENTIMIENTO"
    out.loc[out["en_dnc"], "motivo_no_contacto"] = "EN_DNC"
    return out

In [ ]:
os.environ.setdefault("PHONE_HMAC_KEY", "clave-solo-para-demo-local")
CLAVE = obtener_clave_hmac(CFG)

consentimientos = pd.read_csv(Path(DIR_ENTRADA) / "consentimientos.csv", dtype=str)
dnc = pd.read_csv(Path(DIR_ENTRADA) / "dnc.csv", dtype=str)

silver = aplicar_consentimiento(silver, consentimientos, dnc, CFG)
silver["telefono_hash"] = silver["e164"].map(lambda x: seudonimizar(x, CLAVE))
silver["telefono_enmascarado"] = silver["e164"].map(enmascarar)

silver["motivo_no_contacto"].value_counts().to_frame("filas")

## 5. Deduplicación y *golden record*

In [ ]:
PESOS_VERIFICACION = {
    "verificado": 2.0,
    "presunto": 0.0,
    "sospechoso": -1.0,
    "inactivo": -3.0,
}

def puntuar_registro(df: pd.DataFrame) -> pd.Series:
    completitud = df.notna().sum(axis=1) / df.shape[1]
    recencia = (
        1 - (df["dias_desde_consentimiento"].fillna(10_000).clip(0, 3650) / 3650)
    )
    if "estado_verificado" in df.columns:
        verificacion = df["estado_verificado"].map(PESOS_VERIFICACION).fillna(0.0)
    else:
        verificacion = 0.0

    return (
        df["apto_contacto"].astype(int) * 1000
        + verificacion * 150
        + recencia * 100
        + (10 - df["_prioridad_fuente"].clip(0, 9)) * 10
        + (df["tipo_linea"] == "MOBILE").astype(int) * 5
        + completitud
    )

def deduplicar(df: pd.DataFrame, cfg: ConfigPipeline) -> pd.DataFrame:
    validos = df[df["valido"]].copy()
    validos["_puntaje"] = puntuar_registro(validos)
    validos = validos.sort_values(
        ["cliente_id", "e164", "_puntaje", "_hash_fila"], ascending=[True, True, False, True]
    )

    ganadores = validos.drop_duplicates(subset=["cliente_id", "e164"], keep="first").copy()
    ganadores["fuentes_coincidentes"] = (
        validos.groupby(["cliente_id", "e164"])["_fuente"]
        .agg(lambda s: ",".join(sorted(set(s))))
        .reindex(pd.MultiIndex.from_frame(ganadores[["cliente_id", "e164"]]))
        .to_numpy()
    )

    ganadores["rango_telefono"] = (
        ganadores.sort_values("_puntaje", ascending=False)
        .groupby("cliente_id")
        .cumcount()
        + 1
    )
    ganadores = ganadores[ganadores["rango_telefono"] <= cfg.max_telefonos_por_cliente]
    ganadores["es_principal"] = ganadores["rango_telefono"] == 1

    log.info(
        "Dedup: %d candidatos válidos -> %d registros gold (%d clientes únicos)",
        len(validos), len(ganadores), ganadores["cliente_id"].nunique(),
    )
    return ganadores.sort_values(["cliente_id", "rango_telefono"]).reset_index(drop=True)

def incorporar_verificacion(df: pd.DataFrame, ruta_estados: str | Path | None) -> pd.DataFrame:
    salida = df.copy()
    ruta = Path(ruta_estados) if ruta_estados else None
    if ruta is not None and ruta.exists() and "telefono_hash" in salida.columns:
        estados = pd.read_parquet(ruta)[["telefono_hash", "estado_verificado"]]
        salida = salida.merge(estados.drop_duplicates("telefono_hash"), on="telefono_hash", how="left")
        log.info(
            "Verificación incorporada: %d de %d registros con evidencia de contacto",
            int(salida["estado_verificado"].notna().sum()), len(salida),
        )
    else:
        salida["estado_verificado"] = pd.NA
    salida["estado_verificado"] = salida["estado_verificado"].fillna("presunto")
    return salida

COLUMNAS_GOLD = [
    "cliente_id", "e164", "telefono_hash", "telefono_enmascarado", "extension",
    "tipo_linea", "region", "operador", "zona_horaria", "es_principal",
    "rango_telefono", "apto_contacto", "motivo_no_contacto", "consentimiento_vigente",
    "en_dnc", "dias_desde_consentimiento", "estado_verificado",
    "fuentes_coincidentes", "_fuente", "_ingerido_utc",
]

def construir_gold(df: pd.DataFrame, version: str) -> pd.DataFrame:
    gold = df.loc[:, [c for c in COLUMNAS_GOLD if c in df.columns]].copy()
    gold = gold.rename(columns={"_fuente": "fuente_ganadora", "_ingerido_utc": "ingerido_utc"})
    gold["version_dataset"] = version
    gold["fecha_proceso"] = date.today().isoformat()
    return gold

In [ ]:
silver = incorporar_verificacion(silver, RUTA_ESTADOS)
print(silver["estado_verificado"].value_counts().to_frame("registros").to_string())

gold = construir_gold(deduplicar(silver, CFG), VERSION_DATASET)
print(f"Filas gold: {len(gold)} | clientes: {gold['cliente_id'].nunique()}")
gold.head(10)

## 6. Contrato de datos (Pandera)

In [ ]:
import pandera.pandas as pa
from pandera.pandas import Check, Column, DataFrameSchema

RE_E164 = r"^\+[1-9]\d{7,14}$"

ESQUEMA_GOLD = DataFrameSchema(
    {
        "cliente_id": Column(str, nullable=False, unique=False,
                             checks=Check.str_matches(r"^[A-Za-z0-9\-_]{1,40}$")),
        "e164": Column(str, nullable=False, checks=Check.str_matches(RE_E164)),
        "telefono_hash": Column(str, nullable=False, checks=Check.str_length(64, 64)),
        "telefono_enmascarado": Column(str, nullable=True),
        "extension": Column(str, nullable=True),
        "tipo_linea": Column(str, nullable=False, checks=Check.isin(list(_TIPOS.values()) + ["DESCONOCIDO"])),
        "region": Column(str, nullable=False, checks=Check.str_length(2, 2)),
        "operador": Column(str, nullable=True),
        "zona_horaria": Column(str, nullable=True),
        "es_principal": Column(bool, nullable=False),
        "rango_telefono": Column(int, nullable=False, checks=Check.in_range(1, 3)),
        "apto_contacto": Column(bool, nullable=False),
        "motivo_no_contacto": Column(str, nullable=False),
        "consentimiento_vigente": Column(bool, nullable=False),
        "en_dnc": Column(bool, nullable=False),
        "dias_desde_consentimiento": Column(float, nullable=True),
        "estado_verificado": Column(str, nullable=True,
                                    checks=Check.isin(list(PESOS_VERIFICACION))),
        "fuentes_coincidentes": Column(str, nullable=True),
        "fuente_ganadora": Column(str, nullable=False),
        "ingerido_utc": Column(str, nullable=False),
        "version_dataset": Column(str, nullable=False),
        "fecha_proceso": Column(str, nullable=False),
    },
    checks=[

        Check(lambda d: ~d.duplicated(subset=["cliente_id", "e164"]).any(),
              error="Duplicados en (cliente_id, e164)"),

        Check(lambda d: d.groupby("cliente_id")["es_principal"].sum().eq(1).all(),
              error="Todo cliente debe tener exactamente un teléfono principal"),
    ],
    strict=True,
    coerce=True,
    name="contrato_telefonos_gold_v1",
)

def validar_contrato(df: pd.DataFrame) -> pd.DataFrame:
    return ESQUEMA_GOLD.validate(df, lazy=True)

In [ ]:
try:
    gold_validado = validar_contrato(gold)
    print("✅ Contrato de datos satisfecho")
except pa.errors.SchemaErrors as exc:
    display(exc.failure_cases.head(20))
    raise

## 7. Métricas de calidad y *quality gates*

In [ ]:
@dataclass
class ResultadoGate:
    nombre: str
    valor: float
    umbral: float
    operador: str
    bloqueante: bool
    aprobado: bool

def _col(df: pd.DataFrame, nombre: str) -> pd.Series:
    if nombre in df.columns:
        return df[nombre]
    return pd.Series([pd.NA] * len(df), index=df.index, dtype="object")

def _prop(serie: pd.Series) -> float:
    if serie is None or len(serie) == 0:
        return 0.0
    valor = serie.fillna(False).astype(bool).mean()
    return 0.0 if pd.isna(valor) else round(float(valor), 4)

def _n(serie: pd.Series) -> float:
    if serie is None or len(serie) == 0:
        return 0.0
    return float(int(serie.fillna(False).astype(bool).sum()))

def calcular_metricas(
    bronze: pd.DataFrame, gold: pd.DataFrame, cfg: ConfigPipeline | None = None
) -> dict[str, float]:
    total = max(len(bronze), 1)
    clientes_base = max(int(_col(bronze, "cliente_id").nunique()), 1)
    apto = _col(gold, "apto_contacto").fillna(False).astype(bool) if len(gold) else pd.Series(dtype=bool)
    alcanzables = int(gold.loc[apto.to_numpy(), "cliente_id"].nunique()) if len(gold) and apto.any() else 0
    tasa_valido = _prop(_col(bronze, "valido"))

    metricas: dict[str, float] = {

        "candidatos_bronze": float(total),
        "clientes_base": float(clientes_base),
        "filas_gold": float(len(gold)),
        "clientes_unicos": float(_col(gold, "cliente_id").nunique()),
        "tasa_normalizacion": tasa_valido,
        "tasa_invalidos": round(1 - tasa_valido, 4),
        "tasa_duplicados": round(
            gold.duplicated(subset=["cliente_id", "e164"]).mean()
            if len(gold) and {"cliente_id", "e164"}.issubset(gold.columns) else 0.0, 4
        ),
        "tasa_apto_contacto": _prop(_col(gold, "apto_contacto")),
        "tasa_moviles": round(float((_col(gold, "tipo_linea") == "MOBILE").mean()) if len(gold) else 0.0, 4),
        "tasa_nulos_criticos": round(
            gold[["cliente_id", "e164"]].isna().any(axis=1).mean()
            if len(gold) and {"cliente_id", "e164"}.issubset(gold.columns) else 0.0, 4
        ),
        "cobertura_clientes": round(_col(gold, "cliente_id").nunique() / clientes_base, 4),

        "clientes_alcanzables": float(alcanzables),
        "alcance_contactable": round(alcanzables / clientes_base, 4),
        "telefonos_contactables": _n(_col(gold, "apto_contacto")),

        "tasa_consentimiento_vigente": _prop(_col(gold, "consentimiento_vigente")),
        "registros_en_dnc": _n(_col(gold, "en_dnc")),

        "telefonos_por_cliente": round(
            len(gold) / max(int(_col(gold, "cliente_id").nunique()), 1), 4
        ),
    }

    if len(gold) and {"e164", "cliente_id"}.issubset(gold.columns):
        compartidos = gold.groupby("e164")["cliente_id"].nunique()
        metricas["duplicidad_identidad"] = float(int((compartidos > 1).sum()))
    else:
        metricas["duplicidad_identidad"] = 0.0

    if cfg is not None and "dias_desde_consentimiento" in getattr(gold, "columns", []):
        restantes = cfg.dias_vigencia_consentimiento - pd.to_numeric(
            gold["dias_desde_consentimiento"], errors="coerce"
        )
        for ventana in (30, 60, 90):
            metricas[f"caducan_{ventana}d"] = float(
                int(((restantes >= 0) & (restantes <= ventana)).sum())
            )

    return metricas

_OPERADORES = {
    ">=": lambda v, u: v >= u,
    "<=": lambda v, u: v <= u,
    ">": lambda v, u: v > u,
    "<": lambda v, u: v < u,
    "==": lambda v, u: v == u,
}

def evaluar_gates(metricas: dict[str, float], gates: dict[str, Any]) -> tuple[bool, pd.DataFrame]:
    resultados: list[ResultadoGate] = []
    for nombre, regla in (gates.get("gates") or {}).items():
        if nombre not in metricas:
            raise KeyError(f"El gate '{nombre}' no corresponde a ninguna métrica calculada")
        valor = float(metricas[nombre])
        umbral = float(regla["umbral"])
        op = regla.get("operador", ">=")
        resultados.append(
            ResultadoGate(nombre, valor, umbral, op, bool(regla.get("bloqueante", True)),
                          _OPERADORES[op](valor, umbral))
        )
    tabla = pd.DataFrame([asdict(r) for r in resultados])
    aprobado = bool(tabla.empty or tabla.loc[tabla["bloqueante"], "aprobado"].all())
    return aprobado, tabla

def exigir_gates(aprobado: bool, tabla: pd.DataFrame) -> None:
    if not aprobado:
        fallidos = tabla[~tabla["aprobado"] & tabla["bloqueante"]]
        raise AssertionError(
            "Quality gates bloqueantes no superados:\n" + fallidos.to_string(index=False)
        )

In [ ]:
metricas = calcular_metricas(silver, gold_validado, CFG)
aprobado, tabla_gates = evaluar_gates(metricas, GATES)
display(pd.Series(metricas).to_frame("valor"))
display(tabla_gates)
exigir_gates(aprobado, tabla_gates)
print("✅ Todos los quality gates bloqueantes fueron superados")

## 8. Publicación versionada, promoción y rollback

In [ ]:
def sha256_archivo(ruta: Path) -> str:
    h = hashlib.sha256()
    with open(ruta, "rb") as fh:
        for bloque in iter(lambda: fh.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

def publicar_dataset(
    gold: pd.DataFrame, dir_salida: str | Path, version: str,
    metricas: dict[str, float], ejecucion: dict[str, Any], escribir: bool = True,
) -> dict[str, Any]:
    fecha = date.today().isoformat()
    destino = Path(dir_salida) / f"v={version}" / f"fecha_proceso={fecha}"
    archivo = destino / "telefonos.parquet"

    datos = gold.to_parquet(index=False, compression="snappy")
    if escribir:
        destino.mkdir(parents=True, exist_ok=True)
        archivo.write_bytes(datos)

    manifest = {
        "dataset": "clientes_telefonos_gold",
        "version": version,
        "contrato": ESQUEMA_GOLD.name,
        "ruta": str(archivo.resolve()),
        "sha256": hashlib.sha256(datos).hexdigest(),
        "filas": int(len(gold)),
        "columnas": list(gold.columns),
        "metricas": metricas,
        "ejecucion": ejecucion,
        "publicado_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    if escribir:
        (destino / "manifest.json").write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        log.info("Publicado %s (%d filas, sha256=%s…)", archivo, len(gold), manifest["sha256"][:12])
    else:
        log.info("Modo seco: manifest calculado sin escribir (%d filas)", len(gold))
    return manifest

def promover(dir_salida: str | Path, manifest: dict[str, Any]) -> Path:
    base = Path(dir_salida)
    puntero = base / "current.json"
    if puntero.exists():
        historico = base / "historial_punteros.jsonl"
        with open(historico, "a", encoding="utf-8") as fh:
            fh.write(puntero.read_text(encoding="utf-8").replace("\n", " ") + "\n")
    tmp = base / "current.json.tmp"
    tmp.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(puntero)
    return puntero

def rollback(dir_salida: str | Path, version_objetivo: str) -> dict[str, Any]:
    base = Path(dir_salida)
    candidatos = sorted((base / f"v={version_objetivo}").glob("*/manifest.json"))
    if not candidatos:
        raise FileNotFoundError(f"No existe una publicación para la versión {version_objetivo}")
    manifest = json.loads(candidatos[-1].read_text(encoding="utf-8"))
    if sha256_archivo(Path(manifest["ruta"])) != manifest["sha256"]:
        raise RuntimeError("Integridad comprometida: el SHA-256 no coincide con el manifest")
    promover(base, manifest)
    log.warning("ROLLBACK ejecutado hacia la versión %s", version_objetivo)
    return manifest

In [ ]:
manifest = publicar_dataset(
    gold_validado, DIR_SALIDA, VERSION_DATASET, metricas, EJECUCION, escribir=PUBLICAR
)
if PUBLICAR and ENTORNO == "prod":
    promover(DIR_SALIDA, manifest)

print(json.dumps({k: v for k, v in manifest.items() if k != "columnas"}, indent=2, ensure_ascii=False))
if not PUBLICAR:
    print("\nPUBLICAR=False → manifest calculado en seco; no se escribió ningún artefacto.")

## 9. Emisión al observatorio (Faro)

In [ ]:
import sys

sys.path.insert(0, str(Path("../src").resolve()))

try:
    from faro import Almacen, registrar as registrar_en_faro

    if EMITIR_OBSERVATORIO:
        almacen = Almacen(DIR_OBSERVATORIO)
        escritas = registrar_en_faro(almacen, manifest, silver=silver)
        print("Corrida registrada en el observatorio:", escritas)
        display(almacen.resumen())
    else:
        print("EMITIR_OBSERVATORIO=False → no se alimenta el observatorio.")
except ImportError:
    print("Paquete `faro` no disponible; se omite la emisión al observatorio.")

## 10. Reporte de calidad para el artefacto de CI

In [ ]:
reporte = {
    "metricas": pd.Series(metricas).to_frame("valor").to_html(),
    "gates": tabla_gates.to_html(index=False),
    "motivos": silver["motivo_no_contacto"].value_counts().to_frame("filas").to_html(),
}
html = f"""<h1>Reporte de calidad — teléfonos v{VERSION_DATASET}</h1>
<p>Entorno: <b>{ENTORNO}</b> · commit: <code>{GIT_SHA}</code> · {EJECUCION['inicio_utc']}</p>
<h2>Métricas</h2>{reporte['metricas']}
<h2>Quality gates</h2>{reporte['gates']}
<h2>Motivos de no contacto</h2>{reporte['motivos']}"""
Path("../reports").mkdir(exist_ok=True)
Path("../reports/calidad.html").write_text(html, encoding="utf-8")
print("Reporte escrito en reports/calidad.html")